# W16_M3-2_YOLO实时缺陷 —— 教师版·分步执行（ABCD 全层完成态）

> 形态：与课堂分步一致，每步含【参考答案】与真实执行输出；非聚合脚本。


# AI应用基础（48课时版） · 工单M3-2《YOLO实时缺陷检测》
**第8周 | 2课时(80分钟) | 理实一体**  | 教师参考版（完成态·同构学生版）

| 层 | 任务定位 | 达标要求 |
|----|---------|---------|
| **A 师傅** | 进阶拓展 | 创新优化；示范讲解 |
| **B 组长** | 标准达成 | F1≥0.88 |
| **C 操作员** | 基础完成 | F1≥0.72（2h内） |
| **D 学徒** | 跟做保底 | 跟做示例跑通TODO-1, 程序出结果 |
> 每4周按工单成绩动态升降层（连续2次≥B升一层）



## 一、课前资讯（Predict·预测）
> **预习快问**：["W7课中查①：'fit报JSON错'这个报错怎么排查？", 'W7课中查②：优秀线F1≥0.88你打算怎么冲？']

> **课前检查**：导学单+预习自测已完成；data/数据已就位、starter可运行（--check环境OK）

## 二、PRIMM五步法（先读后写）
本工单按国际编程教学法 **PRIMM** 组织：
1. **Predict 预测**——先运行下方示例，猜输出再验证；
2. **Run 运行**——跑通环境与数据；
3. **Investigate 研究**——读懂starter每段在做什么；
4. **Modify 修改**——完成TODO改造；
5. **Make 创造**——独立完成工单任务。


In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/PycharmProjects/ai-course-teaching-resources/05-AI应用基础48课时版


In [2]:
# --- 教师版运行环境（完成态依赖，勿改）---
import json, re, time          # solutions 完成态所需
import numpy as np
PKG = os.getcwd()              # 课程包根（学生版环境cell已chdir至此）
print("教师版运行环境就绪 · PKG =", PKG)

教师版运行环境就绪 · PKG = /Users/joelgong/PycharmProjects/ai-course-teaching-resources/05-AI应用基础48课时版


## 三、任务要求（Make）
**实施步骤**
1. 错误速查复盘
2. 优化冲刺（B/A层）
3. 三层质检互查
4. 当堂评分与反馈
5. 下次课预告+微课

**验收标准**：及格(D)=`F1≥0.72（2h内）` ；优秀(B+)=`F1≥0.88`


### Starter 1：`# -*- coding: utf-8 -*-`
（Investigate: 读代码→Modify: 补全TODO）

In [3]:
# -*- coding: utf-8 -*-
"""M3-2 实时检测 starter（教学周8）：滑窗+阈值=检测器雏形；FPS与mAP的取舍亲手量

零依赖版检测原理（YOLO思想的极简骨架）：
  图像 → 滑窗扫描 → 每窗"缺陷分" → 阈值 → 定位框 → IoU评估 + FPS计时
真机版(ultralytics YOLOv8n离线权重)见工单——本文件先让你"手搓一个检测器"再谈调参。
"""
import time

import numpy as np

### Starter 2：`def make_scene(seed=7):`
（Investigate: 读代码→Modify: 补全TODO）

In [4]:
def make_scene(seed=7):
    rng = np.random.RandomState(seed)
    img = rng.normal(0.4, 0.05, (64, 64))
    r0, c0, w = 40, 30, 8
    img[r0:r0 + w, c0:c0 + w] += 0.35
    return img, (c0, r0, w, w)

### Starter 3：`def score_map(img, win=8):`
（Investigate: 读代码→Modify: 补全TODO）

In [5]:
def score_map(img, win=8):
    H, W = img.shape[0] // win, img.shape[1] // win
    smap = np.zeros((H, W))
    for r in range(H):
        for c in range(W):
            smap[r, c] = img[r * win:(r + 1) * win, c * win:(c + 1) * win].mean()
    return smap

### 💡 分层提示（先自己试5分钟再看）
**思路提示**
> - TODO-W8-1: 双重循环步长=win，对每个窗取 img[r:r+win,
>   c:c+win].mean() 存入 H×W 分数图
> - TODO-W8-2: 计算交集面积与并集面积之比（4行以内可完成）

| 层 | 这样做 |
|----|--------|
| **D 学徒** | 跟教师演示逐步照做：定位TODO行 → 按上面思路逐句翻译成代码 → 运行看输出 → 对不上就看下方"报错自查" |
| **C 操作员** | 只看思路提示独立实现；卡住再对照教师课堂演示 |
| **B/A 组长·师傅** | 不看提示直接做；完成后做扩展：改参数/换实现对比效果，写进工单"参数-收益"留痕 |

**报错自查三板斧**：①读最后一行错误信息 → ②检查变量名/缩进/括号 → ③还不行问同层同学或教师
**自检**：对照上方"验收标准"，及格(D)=`F1≥0.72（2h内）`，优秀(B+)=`F1≥0.88`

### Starter 4：`def detect(img, win=8, thresh=0.5):`
（Investigate: 读代码→Modify: 补全TODO）

In [6]:
def detect(img, win=8, thresh=0.5):
    smap = score_map(img, win)
    r, c = np.unravel_index(np.argmax(smap), smap.shape)
    if smap[r, c] < thresh:
        return None, float(smap[r, c])
    return (int(c) * win, int(r) * win, win, win), float(smap[r, c])

### Starter 5：`def iou(a, b):`
（Investigate: 读代码→Modify: 补全TODO）

In [7]:
def iou(a, b):
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    ix = max(0, min(ax + aw, bx + bw) - max(ax, bx))
    iy = max(0, min(ay + ah, by + bh) - max(ay, by))
    inter = ix * iy
    union = aw * ah + bw * bh - inter
    return inter / union if union else 0.0

### Starter 6：`def fps_bench(detect_fn, img, n=200):`
（Investigate: 读代码→Modify: 补全TODO）

In [8]:
def fps_bench(fn, img, n=200):
    t0 = time.time()
    for _ in range(n):
        fn(img)
    return n / (time.time() - t0)

### Starter 7：`if __name__ == "__main__":`
（Investigate: 读代码→Modify: 补全TODO）

In [9]:
img, gt = make_scene()

print("=== 工单 M3-2 YOLO实时缺陷检测（零依赖检测器）· 完成态 ===")

print(f"输入: 64×64 产线图 · 真值框(x,y,w,h) = {gt}")

box, conf = detect(img)

v = iou(box, gt)

f = fps_bench(detect, img)

print(f"检测结果: 框={box}  缺陷分={conf:.3f}  IoU={v:.2f}（目标≥0.5）  FPS≈{f:.0f}")

print("\n滑窗尺寸 × 性能取舍表（YOLO调参的本质：感受野 vs 速度 vs 定位精度）:")

print("  win   IoU     FPS")

best_win, best_iou = None, -1

for win in (4, 8, 16):
    b, _ = detect(img, win=win)
    vv = iou(b, gt) if b else 0.0
    ff = fps_bench(lambda im, w=win: detect(im, win=w), img, n=120)
    print(f"  {win:>3}   {vv:.2f}   {ff:>5.0f}")
    if vv > best_iou:
        best_iou, best_win = vv, win

print(f"\n实测最优窗口 win={best_win}（IoU={best_iou:.2f}）—— 窗口过小则单窗信噪比低易误检，"
      f"过大则定位粗化，存在最优值；FPS 随窗口增大而升高（窗数按平方减少）。")

print("结论: 检测器=滑窗+阈值+IoU，与 YOLO 同源；产线要在'毫秒级'约束下实验选优——"
      "这正是锚框尺寸/步长调参的同一问题。")

print("真机路径: ultralytics YOLOv8n 离线权重（data/weights/yolov8n.onnx）→ 同口径 mAP + FPS 复测。")

=== 工单 M3-2 YOLO实时缺陷检测（零依赖检测器）· 完成态 ===
输入: 64×64 产线图 · 真值框(x,y,w,h) = (30, 40, 8, 8)
检测结果: 框=(32, 40, 8, 8)  缺陷分=0.658  IoU=0.60（目标≥0.5）  FPS≈2722

滑窗尺寸 × 性能取舍表（YOLO调参的本质：感受野 vs 速度 vs 定位精度）:
  win   IoU     FPS


    4   0.25     693
    8   0.60    2726
   16   0.00    9976

实测最优窗口 win=8（IoU=0.60）—— 窗口过小则单窗信噪比低易误检，过大则定位粗化，存在最优值；FPS 随窗口增大而升高（窗数按平方减少）。
结论: 检测器=滑窗+阈值+IoU，与 YOLO 同源；产线要在'毫秒级'约束下实验选优——这正是锚框尺寸/步长调参的同一问题。
真机路径: ultralytics YOLOv8n 离线权重（data/weights/yolov8n.onnx）→ 同口径 mAP + FPS 复测。


## 四、提交要求与避坑
- 提交: `学号_姓名_M3-2.py` 或本notebook(含输出)


## A·师傅层：多解对比与帮带要点

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | 窗尺寸调优(win=4/8/16) | 窗小IoU细但FPS降——FPS-mAP三方博弈 |
| B | 阈值调优(thresh) | 只影响检出/漏检，不耗时 |

> 教学点：阈值调参零成本先做；尺寸调参有FPS代价，实测后再定。

---

## 执行与验证

## ABCD 验收对照

- **D·及格/C·良好**：TODO 分步单元（上方）逐个执行通过，输出即证据（0 个参考答案单元）。
- **B·优秀**：标注「(优秀)」的扩展单元已实现并执行。
- **A·师傅**：多解互验与帮带要点见上节；可用于 2 分钟费曼展示。
